# Acte IV : Carte 3D galactique

Nous avons exploré les relations physiques, les familles cachées et les biais instrumentaux. Mais une question demeure. Où se trouvent toutes ces exoplanètes dans la Voie lactée ? Pour y répondre, nous avons construit une carte 3D interactive à partir des coordonnées célestes (ascension droite, déclinaison et distance).

In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

CSV_URL = (
    "https://raw.githubusercontent.com/OpenExoplanetCatalogue/"
    "oec_tables/master/comma_separated/open_exoplanet_catalogue.txt"
)

COLS = [
    "name", "binary_flag", "mass", "radius", "period", "semi_major_axis",
    "eccentricity", "periastron", "longitude", "ascending_node",
    "inclination", "surface_temp", "age", "discovery_method",
    "discovery_year", "last_updated", "ra_string", "dec_string",
    "distance_pc", "host_star_mass", "host_star_radius",
    "host_star_metallicity", "host_star_temp", "host_star_age", "list_flag"
]

df = pd.read_csv(CSV_URL, header=None, comment='#')
df.columns = COLS[:len(df.columns)]

# Faut convertir RA sexagésimal en degrés décimaux
def ra_to_deg(s):
    try:
        h, m, sec = str(s).strip().split()
        return (float(h) + float(m)/60 + float(sec)/3600) * 15
    except:
        return np.nan

# Faut convertir Dec sexagésimal en degrés décimaux
def dec_to_deg(s):
    try:
        s = str(s).strip()
        sign = -1 if s.startswith('-') else 1
        parts = s.lstrip('+-').split()
        d, m, sec = float(parts[0]), float(parts[1]), float(parts[2])
        return sign * (d + m/60 + sec/3600)
    except:
        return np.nan

df["ra_deg"]  = df["ra_string"].apply(ra_to_deg)
df["dec_deg"] = df["dec_string"].apply(dec_to_deg)
df["distance_pc"] = pd.to_numeric(df["distance_pc"], errors="coerce")
df["mass"] = pd.to_numeric(df["mass"], errors="coerce")

df = df.dropna(subset=["ra_deg", "dec_deg", "distance_pc"])
df = df[df["distance_pc"] > 0]

print(f"Planètes chargées : {len(df)}")
print(df["distance_pc"].describe())

# Coordonnées cartésiennes
ra_rad  = np.radians(df["ra_deg"])
dec_rad = np.radians(df["dec_deg"])
r       = np.log1p(df["distance_pc"]) * 18

df["x"] = r * np.cos(dec_rad) * np.cos(ra_rad)
df["y"] = r * np.sin(dec_rad)
df["z"] = r * np.cos(dec_rad) * np.sin(ra_rad)

# Couleurs par méthode de détection
color_map = {
    "transit":         "#4FC3F7",
    "rv":              "#FF8A65",
    "radial velocity": "#FF8A65",
    "imaging":         "#A5D6A7",
    "microlensing":    "#CE93D8",
}

def get_color(method):
    if pd.isna(method):
        return "#888888"
    m = str(method).lower()
    for k, v in color_map.items():
        if k in m:
            return v
    return "#888888"

df["color"] = df["discovery_method"].apply(get_color)
df["size"]  = df["mass"].apply(
    lambda m: min(2.5 + np.log1p(m) * 1.2, 8) if pd.notna(m) else 2.5
)

# Pour le hover
def make_hover(row):
    lines = [f"<b>{row['name']}</b>"]
    lines.append(f"Distance : {row['distance_pc']:.0f} pc")
    if pd.notna(row["discovery_method"]):
        lines.append(f"Détection : {row['discovery_method']}")
    if pd.notna(row["mass"]):
        lines.append(f"Masse : {row['mass']:.2f} MJ")
    if pd.notna(row.get("discovery_year")):
        try:
            lines.append(f"Découverte : {int(row['discovery_year'])}")
        except:
            pass
    return "<br>".join(lines)

df["hover"] = df.apply(make_hover, axis=1)

# Plot 3D
legend_labels = {
    "#4FC3F7": "Transit",
    "#FF8A65": "RV",
    "#A5D6A7": "Imagerie directe",
    "#CE93D8": "Microlentille",
    "#888888": "Autre / inconnu",
}

fig = go.Figure()

for color, label in legend_labels.items():
    sub = df[df["color"] == color]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter3d(
        x=sub["x"], y=sub["y"], z=sub["z"],
        mode="markers",
        name=label,
        marker=dict(size=sub["size"], color=color, opacity=0.85, line=dict(width=0)),
        text=sub["hover"],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers",
    name="Soleil",
    marker=dict(size=4, color="#FFD54F", symbol="diamond"),
    hovertemplate="<b>Soleil</b><br>Notre étoile<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=f"Carte 3D des exoplanètes objets",
        font=dict(size=18, color="white"),
        x=0.5,
    ),
    scene=dict(
        xaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        zaxis=dict(showbackground=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        bgcolor="black",
    ),
    paper_bgcolor="black",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0.6)", bordercolor="rgba(255,255,255,0.2)", borderwidth=1),
    margin=dict(l=0, r=0, t=50, b=0),
    height=750,
)

fig.show()

Planètes chargées : 5186
count    5186.000000
mean      692.805031
std      1136.855900
min         1.295000
25%       102.112500
50%       395.398000
75%       851.979500
max      8500.000000
Name: distance_pc, dtype: float64
